# Problem 3 — Phase 1: Feature representation (SVM)

**Index:** [`hierarchical_problem3_index.ipynb`](hierarchical_problem3_index.ipynb) · **Next:** [Phase 2 SVD](hierarchical_problem3_phase2_truncated_svd.ipynb)

Compare **TF-IDF**, **TF (L2, no IDF)**, and **count** features. **TF-IDF + `LinearSVC` is not retrained here** — it is taken from the same full-tree run as **[`hierarchical_1b.ipynb`](hierarchical_1b.ipynb)**:

- After you run the full-tree training cell in **1b**, it writes **`full_tree_rows.pkl`** in the Homework 4 folder (next to `topics.csv`). That file holds one summary dict per model; we **load the row where `model == "LinearSVC"`** (TF-IDF + SVM, `max_features=8000` in 1b).
- This notebook only **trains** `tf_l2` and `count` (still slow). **`RUN_EXPENSIVE`** controls those two only.

- **Training cell:** loads TF-IDF baseline from pickle, then optional verbose fits for the other two.
- **Summary cell:** combined table + `problem3_phase1_results.csv`.


In [8]:
from __future__ import annotations

import time
from pathlib import Path

import pandas as pd
from IPython.display import display

from hierarchical_training_data import make_multilabel_binary_pool_data
from topic_hierarchy import load_topic_tree, summary


def homework4_base() -> Path:
    here = Path.cwd().resolve()
    if (here / "topics.csv").exists():
        return here
    for p in [here, *here.parents]:
        if (p / "topics.csv").exists():
            return p
    raise FileNotFoundError("topics.csv not found — cwd should be Homework 4")


BASE = homework4_base()
tree = load_topic_tree(str(BASE / "topics.csv"))
mldata = make_multilabel_binary_pool_data(base_path=str(BASE))
SPLIT = "test"
RESTRICT = True
MAX_FEATURES = 8000

print("BASE", BASE)
print(summary(tree))
print("train", len(mldata.train_ids()), "test", len(mldata.test_ids()))

import pickle

from hierarchical_classifier import svm_linear_binary_edge_factory
from hierarchical_summary_metrics import comparison_table, train_full_tree_and_summarize

# Same artifact as hierarchical_1b full-tree training cell
FULL_TREE_ROWS_PKL = BASE / "full_tree_rows.pkl"


BASE /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4
{'traversal_root': 'Root', 'n_nodes': 117, 'n_leaves': 82, 'n_branching_nodes': 22, 'n_local_classifiers': 22, 'n_binary_edge_classifiers': 103, 'max_branching_factor': 23, 'max_depth_from_root': 5}
train 14465 test 3617


In [ ]:
# --- TF-IDF row: from hierarchical_1b (full_tree_rows.pkl), not retrained ---
def load_tfidf_linear_svc_row_from_1b() -> dict:
    if not FULL_TREE_ROWS_PKL.is_file():
        raise FileNotFoundError(
            f"Missing {FULL_TREE_ROWS_PKL}. Run the full-tree training cell in hierarchical_1b.ipynb "
            "first (it saves full_tree_rows.pkl)."
        )
    with open(FULL_TREE_ROWS_PKL, "rb") as f:
        full_tree_rows = pickle.load(f)
    for r in full_tree_rows:
        if r.get("model") == "LinearSVC":
            out = dict(r)
            out["model"] = "SVM_tfidf"
            out["representation"] = "tfidf"
            out["source"] = "hierarchical_1b full_tree_rows.pkl (LinearSVC)"
            return out
    raise KeyError(
        'No row with model=="LinearSVC" in pickle — expected 1b linear_model_factories order.'
    )


rows_p1 = [load_tfidf_linear_svc_row_from_1b()]
print("Loaded TF-IDF baseline from 1b:", FULL_TREE_ROWS_PKL)
# Snapshot for other notebooks / reports (no large matrix columns)
_bad = ("ft_confusion_matrix_pooled", "ft_per_edge_confusion")
_baseline_csv = BASE / "problem3_phase1_baseline_tfidf_from_1b.csv"
pd.DataFrame([{k: v for k, v in rows_p1[0].items() if k not in _bad}]).to_csv(_baseline_csv, index=False)
print("Wrote", _baseline_csv)

# --- Train only tf_l2 and count (same max_features as 1b LinearSVC) ---
RUN_EXPENSIVE = 1  # set True to fit tf_l2 + count (still very slow)

phase1_train_specs = [
    ("SVM_tf_l2", svm_linear_binary_edge_factory(max_features=MAX_FEATURES, text_vectorizer="tf_l2")),
    ("SVM_count", svm_linear_binary_edge_factory(max_features=MAX_FEATURES, text_vectorizer="count")),
]

if RUN_EXPENSIVE:
    for mi, (name, factory) in enumerate(phase1_train_specs, start=1):
        print("\n" + "=" * 72, flush=True)
        print(f"PHASE 1 train [{mi}/{len(phase1_train_specs)}] {name}", flush=True)
        print("=" * 72 + "\n", flush=True)
        t0 = time.perf_counter()
        _, row = train_full_tree_and_summarize(
            name,
            tree,
            mldata,
            factory,
            split=SPLIT,
            verbose=True,
            restrict_to_parent_subtree=RESTRICT,
        )
        row["wall_time_sec"] = time.perf_counter() - t0
        row["representation"] = name.split("_", 1)[-1]
        row["source"] = "trained in phase1 notebook"
        rows_p1.append(row)
        print(
            f"\n>>> {name} finished in {row['wall_time_sec']:.1f}s wall time\n",
            flush=True,
        )
else:
    print(
        "SKIP tf_l2/count training (RUN_EXPENSIVE=False). Summary cell will show TF-IDF only unless you train."
    )


Loaded TF-IDF baseline from 1b: /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/full_tree_rows.pkl
Wrote /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/problem3_phase1_baseline_tfidf_from_1b.csv

PHASE 1 train [1/2] SVM_tf_l2

[1/103] Root → CCAT  (depth 0)


    fit: n=14465  positives=6540
[2/103] Root → ECAT  (depth 0)
    fit: n=14465  positives=4496
[3/103] Root → GCAT  (depth 0)
    fit: n=14465  positives=6402
[4/103] Root → MCAT  (depth 0)
    fit: n=14465  positives=2035
[5/103] CCAT → C1  (depth 1)
    fit: n=6540  positives=4025
[6/103] CCAT → C2  (depth 1)
    fit: n=6540  positives=1288
[7/103] CCAT → C3  (depth 1)
    fit: n=6540  positives=2054
[8/103] CCAT → C4  (depth 1)
    fit: n=6540  positives=806
[9/103] ECAT → E1  (depth 1)
    fit: n=4496  positives=2084
[10/103] ECAT → E2  (depth 1)
    fit: n=4496  positives=765
[11/103] ECAT → E3  (depth 1)
    fit: n=4496  positives=345
[12/103] ECAT → E4  (depth 1)
    fit: n=4496  positives=753
[13/103] ECAT → E5  (depth 1)
    fit: n=4496  positives=872
[14/103] ECAT → E6  (depth 1)
    fit: n=4496  positives=149
[15/103] ECAT → E7  (depth 1)
    fit: n=4496  positives=168
[16/103] GCAT → G1  (depth 1)
    fit: n=6402  positives=1499
[17/103] GCAT → GCRIM  (depth 1)
    fit: n

In [ ]:
# Summary table (run after the training cell — uses `rows_p1` from above)
df_p1 = comparison_table(rows_p1) if rows_p1 else pd.DataFrame()
if df_p1.empty:
    print("No rows in rows_p1 — run the training cell with RUN_EXPENSIVE=True first.")
else:
    bad = {"ft_confusion_matrix_pooled", "ft_per_edge_confusion"}
    cols = [c for c in df_p1.columns if c not in bad and pd.api.types.is_numeric_dtype(df_p1[c])]
    summary_cols = [
        "model",
        "representation",
        "wall_time_sec",
        "ft_pooled_micro_f1",
        "ft_macro_f1",
        "ft_pos_weighted_f1",
        "leaf_macro_f1",
        "leaf_micro_f1",
    ]
    show = [c for c in summary_cols if c in df_p1.columns]
    if "model" in df_p1.columns:
        rest = [c for c in cols if c not in show and c != "model"]
        display(df_p1[show + rest].round(4))
    else:
        display(df_p1.round(4))
    out_path = BASE / "problem3_phase1_results.csv"
    df_save = df_p1[[c for c in df_p1.columns if c not in bad]]
    df_save.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
df_p1

,model,representation,ft_pooled_micro_f1,ft_macro_f1,ft_pos_weighted_f1,leaf_macro_f1,leaf_micro_f1,ft_n_edges_scored,ft_pooled_micro_precision,ft_pooled_micro_recall,...,ft_pos_weighted_precision,ft_pos_weighted_recall,leaf_weighted_f1,leaf_samples_f1,path_gold_branch_recall,n_path_gold_branches,n_path_gold_branches_correct,fit_n_specs,fit_n_edges_trained,fit_n_skipped_single_class
0,SVM_tfidf,tfidf,0.8351,0.7976,0.83,0.6233,0.6455,99.0,0.8797,0.7947,...,0.877,0.7947,0.6496,0.5677,0.8004,13631.0,10910.0,103.0,101.0,2.0


Saved: /Users/nikhileshbelulkar/Documents/HW-Spring-2026/Financial data science and computing/Homework 4/problem3_phase1_results.csv


,model,ft_pooled_micro_f1,ft_macro_f1,ft_pos_weighted_f1,ft_n_edges_scored,ft_pooled_micro_precision,ft_pooled_micro_recall,ft_pooled_micro_accuracy,ft_macro_precision,ft_macro_recall,...,leaf_weighted_f1,leaf_samples_f1,path_gold_branch_recall,n_path_gold_branches,n_path_gold_branches_correct,fit_n_specs,fit_n_edges_trained,fit_n_skipped_single_class,representation,source
0,SVM_tfidf,0.835074,0.797626,0.829966,99.0,0.879735,0.794728,0.942962,0.878509,0.741733,...,0.649591,0.567731,0.800381,13631.0,10910.0,103.0,101.0,2.0,tfidf,hierarchical_1b full_tree_rows.pkl (LinearSVC)
